In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
import copy
import random
import os
import matplotlib.pyplot as plt

# ==========================================
# 0. DETERMINISTIC SEEDING
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
# ==========================================
# 1. DATA LOADING & PREPROCESSING
# ==========================================
print("Loading and preprocessing data...")

SPEED_PATH = './data/speed_matrix_2015_5mph'
speed_df = pd.read_pickle(SPEED_PATH)
speed_matrix_np = speed_df.to_numpy()

data_min = np.min(speed_matrix_np)
data_max = np.max(speed_matrix_np)

speed_matrix_scaled = (speed_matrix_np - data_min) / (data_max - data_min)

def create_sliding_windows(data, window_size=12, horizon=1):

    x_offsets = np.arange(window_size)
    y_offsets = np.arange(window_size, window_size + horizon)

    num_samples = data.shape[0] - window_size - horizon + 1

    X, Y = [], []

    for t in range(num_samples):
        X.append(data[t + x_offsets, :])
        Y.append(data[t + y_offsets, :])

    return torch.tensor(np.array(X), dtype=torch.float32), torch.tensor(np.array(Y), dtype=torch.float32)

print("Generating sliding windows...")

X_full, Y_full = create_sliding_windows(speed_matrix_scaled)

def apply_missing_data_mask(X, missing_rate):

    if missing_rate == 0.0:
        return X

    mask = (torch.rand(X.shape) > missing_rate).float()

    return X * mask


rates = [0.0, 0.1, 0.2, 0.4, 0.8]

scenarios = {f"missing_{int(r*100)}": apply_missing_data_mask(X_full, r) for r in rates}

In [ ]:
# ==========================================
# 2. MODEL BLOCKS
# ==========================================
class BaseLSTM(nn.Module):

    def __init__(self, in_dim, hid_dim):
        super().__init__()
        self.lstm = nn.LSTM(in_dim, hid_dim, batch_first=True)

    def forward(self, x):
        return self.lstm(x)[0]


class BaseConvLSTM(nn.Module):

    def __init__(self, in_dim, hid_dim):
        super().__init__()

        self.conv = nn.Conv1d(in_channels=in_dim, out_channels=hid_dim, kernel_size=3, padding=1)

        self.lstm = nn.LSTM(hid_dim, hid_dim, batch_first=True)

    def forward(self, x):

        x = x.permute(0,2,1)
        x = F.relu(self.conv(x))
        x = x.permute(0,2,1)

        return self.lstm(x)[0]


BLOCKS = {
    "LSTM": BaseLSTM,
    "ConvLSTM": BaseConvLSTM
}


In [ ]:
# ==========================================
# 3. GAN MODELS
# ==========================================
class TrafficGenerator(nn.Module):

    def __init__(self, block_type, nodes=323, hid_dim=64):

        super().__init__()

        self.encoder = BLOCKS[block_type](nodes, hid_dim)

        self.decoder = nn.Linear(hid_dim, nodes)

    def forward(self, x):

        features = self.encoder(x)

        return self.decoder(features[:, -1, :]).unsqueeze(1)


class TrafficDiscriminator(nn.Module):

    def __init__(self, block_type, nodes=323, hid_dim=32):

        super().__init__()

        self.encoder = BLOCKS[block_type](nodes, hid_dim)

        self.classifier = nn.Linear(hid_dim, 1)

    def forward(self, sequence):

        features = self.encoder(sequence)

        return torch.sigmoid(self.classifier(features[:, -1, :]))

In [ ]:
# ==========================================
# 4. LOSS FUNCTIONS
# ==========================================
def masked_mse_loss(preds, targets, null_val=0.0):

    mask = (targets != null_val).float()

    loss = (preds - targets) ** 2

    return torch.mean(loss * mask) / (torch.mean(mask) + 1e-5)


def calculate_metrics(preds, targets, null_val=0.0):

    mask = (targets != null_val).float()

    mask /= torch.mean(mask)

    mask = torch.where(torch.isnan(mask), torch.zeros_like(mask), mask)

    mae_loss = torch.abs(preds - targets) * mask
    mae = torch.mean(torch.where(torch.isnan(mae_loss), torch.zeros_like(mae_loss), mae_loss))

    rmse_loss = ((preds - targets) ** 2) * mask
    rmse = torch.sqrt(torch.mean(torch.where(torch.isnan(rmse_loss), torch.zeros_like(rmse_loss), rmse_loss)))

    mape_loss = torch.abs((preds - targets) / (targets + 1e-5)) * mask
    mape = torch.mean(torch.where(torch.isnan(mape_loss), torch.zeros_like(mape_loss), mape_loss)) * 100

    return mae.item(), rmse.item(), mape.item()


adv_criterion = nn.BCELoss()
LAMBDA_MSE = 100

# ==========================================
# EARLY STOPPING
# ==========================================
class GeneratorEarlyStopping:

    def __init__(self, patience=5):

        self.patience = patience
        self.counter = 0
        self.best_loss = float('inf')
        self.early_stop = False
        self.best_state = None

    def __call__(self, val_mse, model):

        if val_mse < self.best_loss - 1e-5:

            self.best_loss = val_mse
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())

        else:

            self.counter += 1

            if self.counter >= self.patience:
                self.early_stop = True


In [ ]:
# ==========================================
# 5. TRAINING LOOP (ONLY LSTM + CONVLSTM)
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

epochs = 15
BATCH_SIZE = 128

generator_type = "LSTM"
discriminator_type = "ConvLSTM"

scenario_predictions = {}
scenario_targets = []
results = []

for exp_name, X_masked in scenarios.items():

    print(f"\nSCENARIO: {exp_name}")

    split_idx = int(len(X_masked) * 0.8)

    X_train = X_masked[:split_idx].to(device)
    Y_train = Y_full[:split_idx].to(device)

    X_test = X_masked[split_idx:].to(device)
    Y_test = Y_full[split_idx:].to(device)

    netG = TrafficGenerator(generator_type).to(device)
    netD = TrafficDiscriminator(discriminator_type).to(device)

    optG = torch.optim.Adam(netG.parameters(), lr=0.001)
    optD = torch.optim.Adam(netD.parameters(), lr=0.001)

    early_stopper = GeneratorEarlyStopping(patience=3)

    for epoch in range(epochs):

        netG.train()
        netD.train()

        for i in range(0, len(X_train), BATCH_SIZE):

            batch_X = X_train[i:i+BATCH_SIZE]
            batch_Y = Y_train[i:i+BATCH_SIZE]

            real_labels = torch.ones(batch_X.size(0),1).to(device)
            fake_labels = torch.zeros(batch_X.size(0),1).to(device)

            # Train Discriminator
            optD.zero_grad()

            real_seq = torch.cat([batch_X, batch_Y], dim=1)
            pred_real = netD(real_seq)
            lossD_real = adv_criterion(pred_real, real_labels)

            fake_step = netG(batch_X)
            fake_seq = torch.cat([batch_X, fake_step.detach()], dim=1)

            pred_fake = netD(fake_seq)
            lossD_fake = adv_criterion(pred_fake, fake_labels)

            lossD = lossD_real + lossD_fake
            lossD.backward()
            optD.step()

            # Train Generator
            optG.zero_grad()

            fake_seq_G = torch.cat([batch_X, fake_step], dim=1)

            pred_fake_G = netD(fake_seq_G)

            lossG_adv = adv_criterion(pred_fake_G, real_labels)

            lossG_mse = masked_mse_loss(fake_step, batch_Y)

            lossG = lossG_adv + (LAMBDA_MSE * lossG_mse)

            lossG.backward()
            optG.step()

        # Validation
        netG.eval()

        with torch.no_grad():

            val_fake = netG(X_test[:512])

            val_mse = masked_mse_loss(val_fake, Y_test[:512])

        early_stopper(val_mse.item(), netG)

        if early_stopper.early_stop:
            break

    # Final Evaluation
    netG.load_state_dict(early_stopper.best_state)
    netG.eval()

    with torch.no_grad():

        preds_scaled = netG(X_test[:1024])

        preds = (preds_scaled * (data_max - data_min)) + data_min
        targets = (Y_test[:1024] * (data_max - data_min)) + data_min

        mae, rmse, mape = calculate_metrics(preds, targets)

        pred_std = torch.std(preds).item()
        true_std = torch.std(targets).item()
        std_ratio = pred_std / (true_std + 1e-8)

        results.append([exp_name, mae, rmse, mape, true_std, pred_std, std_ratio])

        scenario_predictions[exp_name] = preds.cpu().numpy()
        scenario_targets.append(targets.cpu().numpy())


In [ ]:
# ==========================================
# RESULTS TABLE
# ==========================================
results_df = pd.DataFrame(
    results,
    columns=[
        "Scenario",
        "MAE",
        "RMSE",
        "MAPE",
        "GT_STD",
        "Pred_STD",
        "STD_Ratio"
    ]
)

print(results_df)

results_df.to_csv("lstm_convlstm_results.csv", index=False)

# ==========================================
# TRAFFIC WAVE PLOTS
# ==========================================
sensor_idx = 10
time_window = 2000

fig, axes = plt.subplots(len(scenarios),1, figsize=(18,12))

for i, scene in enumerate(scenario_predictions):

    preds = scenario_predictions[scene]
    targets = scenario_targets[i]

    gt = targets[:time_window,0,sensor_idx]
    pr = preds[:time_window,0,sensor_idx]

    axes[i].plot(gt, label="Ground truth")
    axes[i].plot(pr, label="Predicted value")

    axes[i].set_title(scene)
    axes[i].set_ylabel("Speed (mph)")
    axes[i].grid(True)

axes[-1].set_xlabel("Time (5-min intervals)")
axes[0].legend()

plt.tight_layout()
plt.savefig("traffic_prediction_scenarios.png", dpi=300)
plt.show()

# ==========================================
# STD COMPARISON GRAPH
# ==========================================
plt.figure(figsize=(10,6))

scenes = results_df["Scenario"]

gt_std = results_df["GT_STD"]
pred_std = results_df["Pred_STD"]

x = np.arange(len(scenes))

plt.bar(x-0.2, gt_std, width=0.4, label="Ground Truth STD")
plt.bar(x+0.2, pred_std, width=0.4, label="Predicted STD")

plt.xticks(x, scenes)

plt.ylabel("Standard Deviation")

plt.title("Traffic Variability Capture")

plt.legend()

plt.grid(True, alpha=0.3)

plt.savefig("std_comparison.png", dpi=300)

plt.show()